# Forecast Evaluation of GMST Models Through 2025

This notebook evaluates how well annual global-mean surface temperature forecasts perform over 2011-2025 when we use model structures inspired by Foster-Rahmstorf (2011) and a forcing-based reduced-complexity setup.

## Research Design

We compare five model families on an expanding-window forecast exercise for 2011-2025:

1. **Foster-Rahmstorf style baseline**: linear trend plus ENSO, volcanic, and solar terms.
2. **Foster-Rahmstorf with AR corrections**: the same baseline plus AR(1), AR(2), and AR(3) residual structure.
3. **Reduced-complexity forcing model**: a transparent forcing-driven regression.
4. **Extended statistical model**: extra forcings, nonlinearities, and interaction terms.
5. **Foster-Rahmstorf `statsmodels` analogue**: lag-searched driver terms using the paper-style regression structure, fitted inside `statsmodels`.

Important: Foster-Rahmstorf (2011) is a **monthly** model. The repo data used here are currently **annual**, so the new `statsmodels` block is an annual analogue with lag search, not a paper-exact monthly replication.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.ar_model import AutoReg

plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data_forecasts"
RESULTS_DIR = DATA_DIR / "results"
SCRIPTS_DIR = ROOT / "scripts"

DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

try:
    from foster_rahmstorf_statsmodels import (
        build_design_matrix,
        effective_sample_size_ar1,
        fit_residual_arma11,
        search_foster_rahmstorf_lags,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing scripts/foster_rahmstorf_statsmodels.py. Run this notebook from the repo root."
    ) from exc

TRAIN_END = 2010
TEST_START = 2011
TEST_END = 2025

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print(f"Working directory: {ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Forecast evaluation window: {TEST_START}-{TEST_END}")

## Step 1: Prepare Comparable Yearly Datasets

The notebook expects these files in `data_forecasts/`:

- `gmst.csv`: `year`, `temp`
- `fr_drivers.csv`: `year`, `enso`, `volcanic`, `solar`
- `forcings.csv`: `year`, `anthro_erf`, `ghg_erf`, `aerosol_erf`, `solar_erf`, `volcanic_erf`, `ch4_erf`, `ods_erf`

If they are missing or empty, run `python scripts/build_forecast_inputs.py` from the repo root, then rerun the loading cell below.

In [ ]:
def create_template_files(start=1970, end=2025):
    years = np.arange(start, end + 1)
    templates = {
        "gmst.csv": pd.DataFrame({"year": years, "temp": np.nan}),
        "fr_drivers.csv": pd.DataFrame(
            {
                "year": years,
                "enso": np.nan,
                "volcanic": np.nan,
                "solar": np.nan,
            }
        ),
        "forcings.csv": pd.DataFrame(
            {
                "year": years,
                "anthro_erf": np.nan,
                "ghg_erf": np.nan,
                "aerosol_erf": np.nan,
                "solar_erf": np.nan,
                "volcanic_erf": np.nan,
                "ch4_erf": np.nan,
                "ods_erf": np.nan,
            }
        ),
    }

    for filename, frame in templates.items():
        path = DATA_DIR / filename
        if not path.exists():
            frame.to_csv(path, index=False)
            print(f"Created template: {path}")

def load_csv(filename, required_columns):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run python scripts/build_forecast_inputs.py or add the dataset manually."
        )

    frame = pd.read_csv(path)
    frame.columns = [str(col).strip().lower() for col in frame.columns]

    missing = sorted(set(required_columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{filename} is missing required columns: {missing}")

    frame = frame.loc[:, ~frame.columns.duplicated()].copy()
    frame["year"] = frame["year"].astype(int)
    return frame.sort_values("year").reset_index(drop=True)

def load_inputs():
    gmst = load_csv("gmst.csv", ["year", "temp"])
    fr = load_csv("fr_drivers.csv", ["year", "enso", "volcanic", "solar"])
    forcings = load_csv(
        "forcings.csv",
        [
            "year",
            "anthro_erf",
            "ghg_erf",
            "aerosol_erf",
            "solar_erf",
            "volcanic_erf",
            "ch4_erf",
            "ods_erf",
        ],
    )

    data = gmst.merge(fr, on="year", how="inner").merge(forcings, on="year", how="inner")
    data = data[(data["year"] >= 1979) & (data["year"] <= TEST_END)].copy()
    data = data.dropna().reset_index(drop=True)

    if data.empty:
        raise ValueError(
            "Merged dataset is empty after alignment and dropna(). Run python scripts/build_forecast_inputs.py, then rerun this cell."
        )

    return data

def show_template_status():
    status_rows = []
    for filename in ["gmst.csv", "fr_drivers.csv", "forcings.csv"]:
        path = DATA_DIR / filename
        if not path.exists():
            status_rows.append(
                {
                    "file": filename,
                    "rows": 0,
                    "fully_filled_rows": 0,
                    "all_value_columns_empty": True,
                    "exists": False,
                }
            )
            continue

        frame = pd.read_csv(path)
        value_columns = [col for col in frame.columns if col != "year"]
        filled_rows = int(frame[value_columns].notna().all(axis=1).sum()) if value_columns else 0
        status_rows.append(
            {
                "file": filename,
                "rows": len(frame),
                "fully_filled_rows": filled_rows,
                "all_value_columns_empty": bool(frame[value_columns].isna().all().all()) if value_columns else True,
                "exists": True,
            }
        )
    return pd.DataFrame(status_rows)

create_template_files()
data = None

try:
    data = load_inputs()
    display(data.head())
except (FileNotFoundError, ValueError) as exc:
    print(exc)
    print("Input files are missing or empty. Run python scripts/build_forecast_inputs.py, then rerun this cell.")
    display(show_template_status())

## Step 2: Define Model Families and Forecast Helpers

The baseline and forcing-driven models are annual OLS regressions. The extended model adds nonlinearities and interactions. The new `statsmodels` Foster-Rahmstorf helper uses lag search over ENSO, volcanic, and solar drivers on each expanding training window.

In [ ]:
def engineer_features(frame):
    out = frame.copy()
    out["time"] = out["year"] - out["year"].min()
    out["time_sq"] = out["time"] ** 2
    out["enso_sq"] = out["enso"] ** 2
    out["enso_x_volcanic"] = out["enso"] * out["volcanic"]
    out["solar_x_volcanic"] = out["solar_erf"] * out["volcanic_erf"]
    out["ghg_minus_aerosol"] = out["ghg_erf"] - out["aerosol_erf"]
    return out

MODEL_SPECS = {
    "foster_rahmstorf": ["time", "enso", "volcanic", "solar"],
    "forcing_rcm_like": [
        "anthro_erf",
        "solar_erf",
        "volcanic_erf",
        "ch4_erf",
        "ods_erf",
    ],
    "extended": [
        "time",
        "time_sq",
        "enso",
        "enso_sq",
        "volcanic",
        "solar",
        "ghg_erf",
        "aerosol_erf",
        "solar_erf",
        "volcanic_erf",
        "ch4_erf",
        "ods_erf",
        "enso_x_volcanic",
        "solar_x_volcanic",
        "ghg_minus_aerosol",
    ],
}

def fit_ols(train, features):
    x_train = sm.add_constant(train[features], has_constant="add")
    y_train = train["temp"]
    return sm.OLS(y_train, x_train, missing="drop").fit()

def one_step_residual_forecast(residuals, lag_order):
    residuals = pd.Series(residuals).dropna()
    if lag_order <= 0 or len(residuals) <= lag_order + 2:
        return 0.0

    model = AutoReg(residuals, lags=lag_order, old_names=False).fit()
    forecast = model.predict(start=len(residuals), end=len(residuals))
    return float(np.asarray(forecast)[0])

def recursive_forecast(data, features, start_year=TEST_START, end_year=TEST_END, ar_lag=0):
    rows = []

    for year in range(start_year, end_year + 1):
        train = data[data["year"] < year].copy()
        target = data[data["year"] == year].copy()

        if train.empty or target.empty:
            continue

        ols = fit_ols(train, features)
        x_target = sm.add_constant(target[features], has_constant="add")
        base_prediction = float(ols.predict(x_target).iloc[0])

        fitted_train = pd.Series(
            ols.predict(sm.add_constant(train[features], has_constant="add")),
            index=train.index,
        )
        residuals = train["temp"] - fitted_train
        ar_correction = one_step_residual_forecast(residuals, ar_lag)

        rows.append(
            {
                "year": year,
                "actual": float(target["temp"].iloc[0]),
                "predicted": base_prediction + ar_correction,
                "base_prediction": base_prediction,
                "ar_correction": ar_correction,
                "ar_lag": ar_lag,
            }
        )

    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError("No recursive forecasts were produced. Check the year coverage.")
    return out

def recursive_forecast_fr_statsmodels(
    data,
    start_year=TEST_START,
    end_year=TEST_END,
    max_driver_lag=3,
    residual_ar=0,
):
    rows = []
    required = ["year", "temp", "enso", "volcanic", "solar"]

    for year in range(start_year, end_year + 1):
        train = data.loc[data["year"] < year, required].copy().reset_index(drop=True)
        target = data.loc[data["year"] == year, required].copy().reset_index(drop=True)

        if train.empty or target.empty:
            continue

        fit = search_foster_rahmstorf_lags(
            train.drop(columns="year"),
            max_lag=max_driver_lag,
            include_fourier=False,
        )

        scaffold = pd.concat(
            [train.drop(columns="year"), target.drop(columns="year")],
            ignore_index=True,
        )
        design, _, _ = build_design_matrix(
            scaffold,
            lag_map=fit.lags,
            include_fourier=False,
        )
        target_index = scaffold.index[-1]
        if target_index not in design.index:
            continue

        x_target = sm.add_constant(design.loc[[target_index]], has_constant="add")
        base_prediction = float(np.asarray(fit.results.predict(x_target))[0])
        residuals = pd.Series(fit.results.resid)
        ar_correction = one_step_residual_forecast(residuals, residual_ar)
        n_eff, rho1 = effective_sample_size_ar1(residuals)

        rows.append(
            {
                "year": year,
                "actual": float(target["temp"].iloc[0]),
                "predicted": base_prediction + ar_correction,
                "base_prediction": base_prediction,
                "ar_correction": ar_correction,
                "ar_lag": residual_ar,
                "enso_lag": int(fit.lags["enso"]),
                "volcanic_lag": int(fit.lags["volcanic"]),
                "solar_lag": int(fit.lags["solar"]),
                "train_r2": float(fit.results.rsquared),
                "train_aic": float(fit.results.aic),
                "n_eff_ar1": float(n_eff),
                "rho1": float(rho1) if np.isfinite(rho1) else np.nan,
            }
        )

    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError("No Foster-Rahmstorf statsmodels forecasts were produced.")
    return out

if "data" not in globals() or data is None:
    try:
        data = load_inputs()
    except (FileNotFoundError, ValueError) as exc:
        raise RuntimeError(
            "No usable input data is loaded. Run python scripts/build_forecast_inputs.py, rerun the loading cell, then rerun this cell."
        ) from exc

data = engineer_features(data)
display(data.tail())

## Step 2A: Inspect the Foster-Rahmstorf `statsmodels` Analogue

This cell fits the lag-searched annual analogue on the 1979-2010 training window, then reports the selected driver lags, coefficient table, and residual autocorrelation diagnostics.

In [ ]:
if "data" not in globals() or data is None:
    data = engineer_features(load_inputs())

fr_train = data.loc[data["year"] <= TRAIN_END, ["temp", "enso", "volcanic", "solar"]].reset_index(drop=True)
fr_paper_fit = search_foster_rahmstorf_lags(
    fr_train,
    max_lag=3,
    include_fourier=False,
)

arma11 = fit_residual_arma11(fr_paper_fit.results.resid)
n_eff, rho1 = effective_sample_size_ar1(fr_paper_fit.results.resid)

fit_summary = pd.DataFrame(
    [
        {
            "selected_enso_lag": fr_paper_fit.lags["enso"],
            "selected_volcanic_lag": fr_paper_fit.lags["volcanic"],
            "selected_solar_lag": fr_paper_fit.lags["solar"],
            "r_squared": float(fr_paper_fit.results.rsquared),
            "adj_r_squared": float(fr_paper_fit.results.rsquared_adj),
            "aic": float(fr_paper_fit.results.aic),
            "arma11_aic": float(arma11.aic),
            "effective_n_ar1": float(n_eff),
            "residual_rho1": float(rho1) if np.isfinite(rho1) else np.nan,
        }
    ]
)

coef_table = pd.DataFrame(
    {
        "term": fr_paper_fit.results.params.index,
        "coefficient": fr_paper_fit.results.params.values,
        "std_error": fr_paper_fit.results.bse.values,
        "p_value": fr_paper_fit.results.pvalues.values,
    }
)

display(fit_summary)
display(coef_table)

## Step 3: Evaluate Forecast Skill and Compare Extensions

The cell below runs annual expanding-window forecasts for 2011-2025, scores each model, saves the tables to `data_forecasts/results/`, and plots the top-ranked forecasts.

In [ ]:
if "data" not in globals() or data is None:
    try:
        data = engineer_features(load_inputs())
    except (FileNotFoundError, ValueError) as exc:
        raise RuntimeError(
            "No usable input data is loaded. Run python scripts/build_forecast_inputs.py, rerun the loading cell, then rerun this cell."
        ) from exc

experiments = [
    {"model": "foster_rahmstorf", "kind": "ols", "features": MODEL_SPECS["foster_rahmstorf"], "ar_lag": 0},
    {"model": "fr_ar1", "kind": "ols", "features": MODEL_SPECS["foster_rahmstorf"], "ar_lag": 1},
    {"model": "fr_ar2", "kind": "ols", "features": MODEL_SPECS["foster_rahmstorf"], "ar_lag": 2},
    {"model": "fr_ar3", "kind": "ols", "features": MODEL_SPECS["foster_rahmstorf"], "ar_lag": 3},
    {
        "model": "fr_statsmodels_lag_search",
        "kind": "fr_statsmodels",
        "max_driver_lag": 3,
        "residual_ar": 0,
    },
    {
        "model": "fr_statsmodels_lag_search_ar1",
        "kind": "fr_statsmodels",
        "max_driver_lag": 3,
        "residual_ar": 1,
    },
    {"model": "forcing_rcm_like", "kind": "ols", "features": MODEL_SPECS["forcing_rcm_like"], "ar_lag": 0},
    {"model": "forcing_ar1", "kind": "ols", "features": MODEL_SPECS["forcing_rcm_like"], "ar_lag": 1},
    {"model": "forcing_ar2", "kind": "ols", "features": MODEL_SPECS["forcing_rcm_like"], "ar_lag": 2},
    {"model": "forcing_ar3", "kind": "ols", "features": MODEL_SPECS["forcing_rcm_like"], "ar_lag": 3},
    {"model": "extended", "kind": "ols", "features": MODEL_SPECS["extended"], "ar_lag": 0},
    {"model": "extended_ar1", "kind": "ols", "features": MODEL_SPECS["extended"], "ar_lag": 1},
    {"model": "extended_ar2", "kind": "ols", "features": MODEL_SPECS["extended"], "ar_lag": 2},
    {"model": "extended_ar3", "kind": "ols", "features": MODEL_SPECS["extended"], "ar_lag": 3},
]

score_rows = []
forecast_frames = []

for config in experiments:
    if config["kind"] == "ols":
        preds = recursive_forecast(data, config["features"], ar_lag=config["ar_lag"])
    else:
        preds = recursive_forecast_fr_statsmodels(
            data,
            max_driver_lag=config["max_driver_lag"],
            residual_ar=config["residual_ar"],
        )

    preds["model"] = config["model"]
    forecast_frames.append(preds)

    score_rows.append(
        {
            "model": config["model"],
            "rmse": rmse(preds["actual"], preds["predicted"]),
            "mae": float(mean_absolute_error(preds["actual"], preds["predicted"])),
            "bias": float((preds["predicted"] - preds["actual"]).mean()),
            "n_forecasts": len(preds),
            "mean_enso_lag": float(preds["enso_lag"].mean()) if "enso_lag" in preds.columns else np.nan,
            "mean_volcanic_lag": float(preds["volcanic_lag"].mean()) if "volcanic_lag" in preds.columns else np.nan,
            "mean_solar_lag": float(preds["solar_lag"].mean()) if "solar_lag" in preds.columns else np.nan,
        }
    )

scores = pd.DataFrame(score_rows).sort_values(["rmse", "mae"]).reset_index(drop=True)
all_forecasts = pd.concat(forecast_frames, ignore_index=True)

scores.to_csv(RESULTS_DIR / "forecast_scores.csv", index=False)
all_forecasts.to_csv(RESULTS_DIR / "all_forecasts.csv", index=False)

display(scores)

top_models = scores.head(6)["model"].tolist()
plot_data = all_forecasts[all_forecasts["model"].isin(top_models)].copy()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

observed = plot_data.groupby("year", as_index=False)["actual"].first()
axes[0].plot(observed["year"], observed["actual"], marker="o", label="Observed", color="black", linewidth=2)
for model_name, frame in plot_data.groupby("model"):
    axes[0].plot(frame["year"], frame["predicted"], marker="o", alpha=0.85, label=model_name)
axes[0].set_title("Observed vs forecast GMST, top models only")
axes[0].set_ylabel("Temperature anomaly")
axes[0].legend(ncol=2)

for model_name, frame in plot_data.groupby("model"):
    axes[1].plot(
        frame["year"],
        frame["predicted"] - frame["actual"],
        marker="o",
        alpha=0.85,
        label=model_name,
    )
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Forecast errors by year, top models only")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Predicted minus observed")
axes[1].legend(ncol=2)

plt.tight_layout()
plt.show()

print(f"Best holdout model by RMSE: {scores.loc[0, 'model']}")
print(f"Saved forecast tables to {RESULTS_DIR}")
print("The lag-searched Foster-Rahmstorf rows report mean selected ENSO, volcanic, and solar lags across the forecast windows.")